# VERITAS Training - Google Colab

Complete training pipeline for VERITAS model on Google Colab with GPU.

**Requirements:**
- Google Colab with GPU runtime (T4 or better)
- Google Drive with dataset mounted
- ~16GB GPU memory (or adjust batch size)

## 1. Setup: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install Dependencies

In [ ]:
# Install required packages
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q opencv-python pillow scipy scikit-learn tqdm matplotlib

## 3. Verify Environment

In [ ]:
import torch
import sys
import os

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU Memory:", torch.cuda.get_device_properties(0).total_memory / 1024**3, "GB")

## 4. Add Project to Path

In [ ]:
# Add src to path (adjust if your code is in a different location)
import sys
sys.path.insert(0, '/content/src')  # If code is in /content/src

# Or if you uploaded to Google Drive:
# sys.path.insert(0, '/content/drive/MyDrive/VERITAS/code/src')

## 5. Import VERITAS Components

In [ ]:
from src.config import Config
from src.models import VERITASModel, MultiTaskLoss
from src.training import Trainer, TrainingConfig, log_experiment
from src.utils.reproducibility import set_random_seed
from src.data.dataset import VERITASDataset
from torch.utils.data import DataLoader

print("✓ All imports successful!")

## 6. Configuration

In [ ]:
# =============================================================================
# CONFIGURATION - Adjust these settings
# =============================================================================

# Dataset paths
DATASET_PATH = '/content/drive/MyDrive/VERITAS/dataset'
CHECKPOINT_DIR = '/content/drive/MyDrive/VERITAS/checkpoints'
LOG_DIR = '/content/drive/MyDrive/VERITAS/logs'

# Model configuration
USE_MULTI_REPRESENTATION = True  # Set False for RGB-only baseline
USE_FULL_SEGMENTATION = True     # Set False for lightweight model

# Training configuration
NUM_EPOCHS = 50
BATCH_SIZE = 4  # Reduce to 2 or 1 if OOM
LEARNING_RATE = 1e-4
RANDOM_SEED = 42

# Early stopping
USE_EARLY_STOPPING = True
EARLY_STOPPING_PATIENCE = 10

print(f"Configuration:")
print(f"  - Multi-representation: {USE_MULTI_REPRESENTATION}")
print(f"  - Full segmentation: {USE_FULL_SEGMENTATION}")
print(f"  - Epochs: {NUM_EPOCHS}")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Learning rate: {LEARNING_RATE}")

## 7. Set Random Seeds

In [ ]:
# Set random seeds for reproducibility
seeds = set_random_seed(RANDOM_SEED)
print(f"Random seeds set: {seeds}")

## 8. Create Data Loaders

In [ ]:
print("Creating data loaders...")

# Model config for dataset
model_config = Config({
    'model': {'input_resolution': 600},
    'representations': {
        'imagenet_mean': [0.485, 0.456, 0.406],
        'imagenet_std': [0.229, 0.224, 0.225]
    }
})

# Create datasets
train_dataset = VERITASDataset(
    annotation_file=f'{DATASET_PATH}/Train/Train_poly.json',
    image_dir=f'{DATASET_PATH}/Train',
    config=model_config,
    transform=None  # TODO: Add augmentation (Task 7)
)

val_dataset = VERITASDataset(
    annotation_file=f'{DATASET_PATH}/Val/Val_poly.json',
    image_dir=f'{DATASET_PATH}/Val',
    config=model_config,
    transform=None
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"✓ Data loaders created")
print(f"  - Train samples: {len(train_dataset)}")
print(f"  - Val samples: {len(val_dataset)}")
print(f"  - Train batches: {len(train_loader)}")
print(f"  - Val batches: {len(val_loader)}")

## 9. Create VERITAS Model

In [ ]:
print("Creating VERITAS model...")

model = VERITASModel(
    config=model_config,
    use_multi_representation=USE_MULTI_REPRESENTATION,
    use_full_segmentation=USE_FULL_SEGMENTATION
)

# Get model info
arch_summary = model.get_architecture_summary()

print(f"✓ Model created")
print(f"  - Backbone: {arch_summary['backbone']}")
print(f"  - Parameters: {arch_summary['total_parameters']:,}")
print(f"  - Size: {arch_summary['model_size_mb']:.2f} MB")
print(f"  - Multi-representation: {arch_summary['multi_representation']}")
print(f"  - Full segmentation: {arch_summary['full_segmentation']}")

## 10. Create Loss Function

In [ ]:
print("Creating multi-task loss...")

loss_fn = MultiTaskLoss(
    cls_weight=0.4,  # Classification loss weight
    seg_weight=0.6   # Segmentation loss weight
)

print("✓ Loss function created")
print(f"  - Classification weight: 0.4")
print(f"  - Segmentation weight: 0.6")

## 11. Create Training Configuration

In [ ]:
print("Creating training configuration...")

training_config = TrainingConfig(
    # Optimizer
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    optimizer_name='adamw',
    
    # Training
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    gradient_clip_max_norm=1.0,
    
    # Scheduler
    use_scheduler=True,
    scheduler_name='reduce_on_plateau',
    scheduler_patience=5,
    scheduler_factor=0.5,
    scheduler_min_lr=1e-7,
    
    # Checkpointing
    checkpoint_dir=CHECKPOINT_DIR,
    save_every_n_epochs=5,
    save_best_only=False,
    best_metric='val_total_loss',
    best_metric_mode='min',
    
    # Logging
    log_every_n_steps=10,
    log_dir=LOG_DIR,
    
    # Reproducibility
    random_seed=RANDOM_SEED,
    
    # Early stopping
    use_early_stopping=USE_EARLY_STOPPING,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    
    # Device
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

print("✓ Training configuration created")

## 12. Log Experiment Information

In [ ]:
print("Logging reproducibility information...")

dataset_info = {
    'dataset_name': 'OpenForensics',
    'dataset_path': DATASET_PATH,
    'split_info': {
        'train_samples': len(train_dataset),
        'val_samples': len(val_dataset),
        'batch_size': BATCH_SIZE
    }
}

repro_logger = log_experiment(
    log_dir=LOG_DIR,
    model=model,
    config=training_config.to_dict(),
    seeds=seeds,
    dataset_info=dataset_info
)

print("✓ Experiment information logged")
print(f"  - Saved to: {LOG_DIR}")

## 13. Create Trainer

In [ ]:
print("Creating trainer...")

trainer = Trainer(
    model=model,
    loss_fn=loss_fn,
    config=training_config,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=None  # Test set not used during training
)

print("✓ Trainer created")
print(f"  - Device: {trainer.device}")
print(f"  - Optimizer: {training_config.optimizer_name}")
print(f"  - Scheduler: {training_config.scheduler_name}")

## 14. Start Training!

In [ ]:
print("\n" + "="*70)
print(f"Starting VERITAS Training - {NUM_EPOCHS} Epochs")
print("="*70 + "\n")

# Train
trainer.fit()

print("\n" + "="*70)
print("Training Complete!")
print("="*70)
print(f"\nBest validation loss: {trainer.best_metric_value:.4f}")
print(f"Checkpoints saved to: {CHECKPOINT_DIR}")
print(f"Logs saved to: {LOG_DIR}")

## 15. Plot Training History

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

# Plot training history
plot_path = Path(LOG_DIR) / "training_history.png"
trainer.plot_history(save_path=str(plot_path))

print(f"Training history plot saved to: {plot_path}")

## 16. Load Best Model

In [ ]:
from src.training import CheckpointManager

print("Loading best model checkpoint...")

checkpoint_mgr = CheckpointManager(CHECKPOINT_DIR)
best_checkpoint = checkpoint_mgr.load_best_checkpoint(
    model=model,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

print(f"✓ Loaded best checkpoint from epoch {best_checkpoint['epoch']}")
print(f"  - Validation loss: {best_checkpoint['metrics']['val_total_loss']:.4f}")

## 17. Save Training Summary

In [ ]:
import json
from datetime import datetime

summary = {
    'timestamp': datetime.now().isoformat(),
    'configuration': {
        'multi_representation': USE_MULTI_REPRESENTATION,
        'full_segmentation': USE_FULL_SEGMENTATION,
        'num_epochs': NUM_EPOCHS,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE
    },
    'results': {
        'best_epoch': best_checkpoint['epoch'],
        'best_val_loss': best_checkpoint['metrics']['val_total_loss'],
        'final_train_loss': trainer.history['train_loss'][-1],
        'final_val_loss': trainer.history['val_loss'][-1]
    },
    'model': {
        'parameters': arch_summary['total_parameters'],
        'size_mb': arch_summary['model_size_mb']
    }
}

summary_path = Path(LOG_DIR) / "training_summary.json"
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"Training summary saved to: {summary_path}")
print("\nSummary:")
print(json.dumps(summary, indent=2))

## Next Steps

After training completes:

1. **Evaluate on Test Set** (Tasks 23-24)
   - Load best checkpoint
   - Run on test set
   - Compute metrics

2. **Run Ablation Studies** (Tasks 28-31)
   - Train RGB-only model
   - Train classification-only model
   - Compare results

3. **Visualize Results** (Task 33)
   - Generate predicted masks
   - Create overlay visualizations
   - Save examples

4. **Deploy** (Tasks 35-37)
   - Create Gradio app
   - Deploy to Hugging Face Spaces

---

**Checkpoints location:** `/content/drive/MyDrive/VERITAS/checkpoints/`

**Logs location:** `/content/drive/MyDrive/VERITAS/logs/`